# Data Understanding (NO<sub>2</sub> Kota Semarang)



## 1.1 Sumber Data

Data NO₂ diperoleh dari **Copernicus / Sentinel-5P** melalui endpoint openEO (`openeo.dataspace.copernicus.eu`) seperti yang terlihat pada skrip Python yang diberikan. Dalam alur yang dipakai:

- Koleksi yang dimuat: `SENTINEL_5P_L2`.
- Rentang temporal: `["2024-09-20", "2024-10-20"]` (contoh pada kode).
- Area of interest (AOI): sebuah poligon kecil di sekitar koordinat longitude ~110.37–110.48 dan latitude ~ -6.94– -7.03.
- Band yang diminta: `NO2` (komponen NO₂ dari produk pemantauan atmosfer Sentinel-5P).
- Agregasi yang dilakukan:
  - `mask` untuk menandai nilai invalid (negatif) sebagai tidak valid,
  - `aggregate_temporal_period(period="day", reducer="mean")` → rata-rata harian,
  - `aggregate_spatial(geometries=aoi, reducer="mean")` → rata-rata spasial di atas AOI,
  - hasil diekspor batch ke CSV.

---

### Area yang di gunakan

![Area Semarang](../image/daerah_semarang.jpg)

### Import Library python yang digunakan

In [1]:
import openeo
import pandas as pd
import matplotlib.pyplot as plt
import os


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\USERB-PC\AppData\Local\Programs\Python\Python39\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\USERB-PC\AppData\Local\Programs\Python\Python39\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\USERB-PC\AppData\Roaming\Python\Python39\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\USERB-PC\AppData\Local\Programs\Python\Python39\lib\site-packages\traitlet

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

### Koneksi ke Copernicus Data Space

In [47]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


### Scraping Data 

In [50]:

aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [
              110.37196761888663,
              -6.939278165866412
            ],
            [
              110.37196761888663,
              -7.031287911059522
            ],
            [
              110.48327164428252,
              -7.031287911059522
            ],
            [
              110.48327164428252,
              -6.939278165866412
            ],
            [
              110.37196761888663,
              -6.939278165866412
            ]
        ]
    ],
}

s5p = connection.load_collection(
    "SENTINEL_5P_L2",
    spatial_extent={
        "west": 110.37196761888663,
        "south": -7.031287911059522,
        "east": 110.48327164428252,
        "north": -6.939278165866412,
    },
    temporal_extent=["2024-01-01", "2024-05-01"],
    bands=["NO2"],
)

def mask_invalid(x):
    return x < 0

s5p_masked = s5p.mask(s5p.apply(mask_invalid))

daily_mean = s5p_masked.aggregate_temporal_period(period="day", reducer="mean")

daily_mean_aoi = daily_mean.aggregate_spatial(geometries=aoi, reducer="mean")

job = daily_mean_aoi.execute_batch(out_format="CSV")

results = job.get_results()
results.download_files("data-copernicus")

for f in os.listdir("data-copernicus"):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join("data-copernicus", f))
        print("File ditemukan:", f)
        break
    
df["date"] = pd.to_datetime(df["date"])

df["month"] = df["date"].dt.to_period("M")

df_monthly = df.groupby("month", as_index=False)["NO2"].mean()
df

0:00:00 Job 'j-2510220831004745a26f9df00f88cb9c': send 'start'
0:00:12 Job 'j-2510220831004745a26f9df00f88cb9c': created (progress 0%)
0:00:18 Job 'j-2510220831004745a26f9df00f88cb9c': queued (progress 0%)
0:00:24 Job 'j-2510220831004745a26f9df00f88cb9c': queued (progress 0%)
0:00:32 Job 'j-2510220831004745a26f9df00f88cb9c': queued (progress 0%)
0:00:43 Job 'j-2510220831004745a26f9df00f88cb9c': queued (progress 0%)
0:00:55 Job 'j-2510220831004745a26f9df00f88cb9c': queued (progress 0%)
0:01:11 Job 'j-2510220831004745a26f9df00f88cb9c': running (progress N/A)
0:01:30 Job 'j-2510220831004745a26f9df00f88cb9c': running (progress N/A)
0:01:55 Job 'j-2510220831004745a26f9df00f88cb9c': running (progress N/A)
0:02:25 Job 'j-2510220831004745a26f9df00f88cb9c': finished (progress 100%)
File ditemukan: timeseries.csv


C:\Users\USERB-PC\AppData\Local\Temp\ipykernel_6968\2936343387.py:63: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["date"].dt.to_period("M")


,date,feature_index,NO2,month
0,2024-01-16 00:00:00+00:00,0,NaN,2024-01
1,2024-01-21 00:00:00+00:00,0,NaN,2024-01
2,2024-01-19 00:00:00+00:00,0,NaN,2024-01
3,2024-01-18 00:00:00+00:00,0,NaN,2024-01
4,2024-01-14 00:00:00+00:00,0,0.000041,2024-01
...,...,...,...,...
117,2024-02-28 00:00:00+00:00,0,NaN,2024-02
118,2024-02-23 00:00:00+00:00,0,NaN,2024-02
119,2024-03-01 00:00:00+00:00,0,NaN,2024-03
120,2024-02-25 00:00:00+00:00,0,NaN,2024-02


## 1.2 Struktur Kolom (kolom yang muncul berdasarkan cuplikan & kode)

Setelah job dieksekusi dan CSV dibaca ke `pandas.DataFrame`, kolom yang tampak pada cuplikan adalah:

- `date`  
  - Tipe: datetime (pada kode lalu dikonversi `pd.to_datetime(df["date"])`).  
  - Format contoh: `2024-10-12 00:00:00+00:00` → termasuk offset zona waktu (`+00:00` / UTC).  
  - Arti: tanggal tengah periode agregasi (rata-rata harian). Karena agregasi `period="day"`, setiap baris mewakili nilai rata-rata NO₂ untuk hari tersebut.

- `feature_index`  
  - Tipe: integer (contoh: `0` pada banyak baris).  
  - Arti kemungkinan: indeks fitur/patch/pixel tile dari koleksi openEO (bila batch menghasilkan beberapa fitur/spatial tiles). Karena kamu meng-`aggregate_spatial` menggunakan satu AOI, nilai ini sering tetap 0. Bila pipeline menghasilkan beberapa spatial features, nilai ini membantu membedakan feature geometry.

- `NO2`  
  - Tipe: float (angka desimal kecil, atau `NaN` bila tidak ada pengukuran valid hari itu).  
  - Arti: nilai rata-rata NO₂ hasil agregasi spasial+temporal untuk AOI pada tanggal tersebut.  
  - Rentang: pada cuplikan terlihat sangat kecil (mis. `0.000033`, `0.000054`), menandakan unit kolom kepadatan (perlu verifikasi metadata).  
  - `NaN`: menandakan tidak ada data valid (mis. semua piksel negatif/invalid pada hari itu, atau tidak ada pengamatan yang lolos mask).


Contoh Table Data :

In [15]:
new_df = pd.read_csv('data-copernicus/timeseries.csv')
new_df = new_df.sort_values(by='date').reset_index(drop=True)
new_df['date'] = pd.to_datetime(new_df['date']).dt.date
new_df

,date,feature_index,NO2
0,2023-12-31,0,NaN
1,2024-01-01,0,NaN
2,2024-01-02,0,NaN
3,2024-01-03,0,NaN
4,2024-01-04,0,0.000015
...,...,...,...
117,2024-04-26,0,NaN
118,2024-04-27,0,NaN
119,2024-04-28,0,0.000050
120,2024-04-29,0,0.000047


## 1.3 Indentifikasi Missing Value

In [16]:
missing_count = new_df.isnull().sum()
missing_percent = (missing_count / len(new_df)) * 100

missing_table = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percent (%)': missing_percent.round(2)
})

missing_table


,Missing Count,Missing Percent (%)
date,0,0.0
feature_index,0,0.0
NO2,66,54.1


Berdasarkan hasil pemeriksaan terhadap dataset yang diperoleh dari platform Copernicus Open Data (Sentinel-5P L2), terdapat tiga kolom utama yaitu:

- `date` — berisi informasi tanggal pengambilan data, tanpa nilai kosong (0 missing value).

- `feature_index` — kolom indeks fitur yang digunakan sistem, tanpa nilai kosong (0 missing value).

- `NO2` — berisi konsentrasi Nitrogen Dioksida (NO₂) rata-rata harian pada area pengamatan, dengan 10 nilai yang hilang (missing values).

Kehilangan data pada kolom NO₂ kemungkinan disebabkan oleh beberapa faktor seperti gangguan atmosfer, awan tebal, atau noise dalam citra satelit Sentinel-5P yang menghambat pembacaan sensor.